# Generate ChimeraX B-factor defattr files for cell entry

Reads `../../results/summaries/entry_all_cells.csv` and writes one ChimeraX `.defattr`
file per protein (alpha and beta).

Steps:
- Drop rows with a null/NA `structure_site`.
- Split into alpha (`structure_site` ending in `_a`) and beta (`_b`).
- Sum `entry in 292_M3_tat_H5 cells` per site.
- Number each site by the numeric part of `structure_site` (e.g. `-1_a` -> `-1`, `2_a` -> `2`).


In [1]:
import pandas as pd

INPUT_CSV = "../../results/summaries/entry_all_cells.csv"
VALUE_COL = "entry in 292_M3_tat_H5 cells"


In [2]:
# Read input
df = pd.read_csv(INPUT_CSV)
print(f"Read {len(df)} rows from {INPUT_CSV}")

# Drop rows where structure_site is null/NA -- not of interest
df = df[df["structure_site"].notna()].copy()
print(f"{len(df)} rows after dropping null structure_site")

# Numeric site number (strip the _a / _b protein suffix)
df["structure_site"] = df["structure_site"].astype(str)
df["site_number"] = df["structure_site"].str.replace(r"_[ab]$", "", regex=True).astype(int)

# Protein label from the suffix
df["protein"] = df["structure_site"].str.extract(r"_([ab])$")[0].map({"a": "alpha", "b": "beta"})
print(df["protein"].value_counts(dropna=False).to_string())


Read 7813 rows from ../../results/summaries/entry_all_cells.csv
7757 rows after dropping null structure_site
protein
beta     3920
alpha    3837


In [3]:
def write_defattr(protein_df, output_file):
    """Sum entry values per site and write a ChimeraX defattr file."""
    site_sums = (
        protein_df.groupby("site_number")[VALUE_COL]
        .sum()
        .reset_index()
        .sort_values("site_number")
    )

    with open(output_file, "w") as f:
        f.write("# ChimeraX defattr file\n")
        f.write("#\n")
        f.write("attribute: binding\n")
        f.write("match mode: any\n")
        f.write("recipient: residues\n")
        f.write("\n")
        for _, row in site_sums.iterrows():
            f.write(f"\t:{int(row['site_number'])}\t{float(row[VALUE_COL]):.6f}\n")

    print(f"Wrote {output_file}: {len(site_sums)} sites")
    print(f"  entry  min={site_sums[VALUE_COL].min():.6f}  "
          f"max={site_sums[VALUE_COL].max():.6f}  "
          f"mean={site_sums[VALUE_COL].mean():.6f}")
    print("  example lines:")
    print(site_sums.head(5).to_string(index=False))
    return site_sums


In [4]:
# Write one defattr file per protein
for protein, output_file in [
    ("alpha", "bfactors_entry_alpha.defattr"),
    ("beta", "bfactors_entry_beta.defattr"),
]:
    print(f"\n=== {protein} ===")
    write_defattr(df[df["protein"] == protein], output_file)



=== alpha ===
Wrote bfactors_entry_alpha.defattr: 196 sites
  entry  min=-96.133000  max=9.554900  mean=-7.932922
  example lines:
 site_number  entry in 292_M3_tat_H5 cells
          -1                     -2.892850
           1                     -2.825420
           2                     -0.770490
           3                      1.755480
           4                     -1.373485

=== beta ===
Wrote bfactors_entry_beta.defattr: 198 sites
  entry  min=-89.809000  max=2.286560  mean=-10.851577
  example lines:
 site_number  entry in 292_M3_tat_H5 cells
           2                    -33.211050
           3                     -4.439635
           4                      0.496995
           5                     -0.482035
           6                     -1.479760
